In [ ]:
# Cell 1: Install dependencies and check GPU
# Run this first. In Colab: Runtime -> Change runtime type -> GPU.
!pip install -q kaggle
!pip install -q timm==0.9.2

import torch
import torchvision
print('Torch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
!nvidia-smi -L || true


In [ ]:
# Cell 2: Try Kaggle download (kmader/skin-cancer-mnist-ham10000)
# To use Kaggle download, upload kaggle.json first:
#   1) In Colab Files panel, upload kaggle.json, or
#   2) Use files.upload() below.
# If you want to skip Kaggle and use Drive only, set use_kaggle = False.
from pathlib import Path
import os

DATA_DIR = Path('/content/HAM10000')
DATA_DIR.mkdir(parents=True, exist_ok=True)
print('DATA_DIR:', DATA_DIR)

use_kaggle = True

# Optional uploader:
# from google.colab import files
# files.upload()

if use_kaggle:
    kaggle_json = Path('/content/kaggle.json')
    if kaggle_json.exists():
        os.environ['KAGGLE_CONFIG_DIR'] = '/content'
        kaggle_json.chmod(0o600)
        !kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p /content/HAM10000 --unzip
        print('Kaggle dataset download attempt finished.')
    else:
        print('kaggle.json not found at /content/kaggle.json.')
        print('Upload kaggle.json and re-run this cell, or set use_kaggle=False and use Drive fallback in Cell 3.')
else:
    print('Skipping Kaggle by user choice (use_kaggle=False). Use Drive fallback in Cell 3.')


In [ ]:
# Cell 3: Google Drive fallback + rsync
# Use this when Kaggle is skipped/unavailable.
# Put dataset under: /content/drive/MyDrive/HAM10000
from pathlib import Path
from google.colab import drive

DRIVE_DATA_DIR = Path('/content/drive/MyDrive/HAM10000')
LOCAL_DATA_DIR = Path('/content/HAM10000')
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

drive.mount('/content/drive')

if DRIVE_DATA_DIR.exists():
    print(f'Found Drive dataset at {DRIVE_DATA_DIR}. Syncing to {LOCAL_DATA_DIR}...')
    !rsync -av --ignore-existing /content/drive/MyDrive/HAM10000/ /content/HAM10000/
    print('Drive sync completed.')
else:
    print(f'No dataset found at {DRIVE_DATA_DIR}. Place files there or update DRIVE_DATA_DIR path.')


In [ ]:
# Cell 4: Data sanity checks
from pathlib import Path

base = Path('/content/HAM10000')
expected = [
    'HAM10000_metadata.csv',
    'HAM10000_images_part_1',
    'HAM10000_images_part_2',
    'HAM10000_images',
]

print('Checking expected files/folders under', base)
for name in expected:
    p = base / name
    print(f'- {name}:', 'FOUND' if p.exists() else 'missing')

print('
Top-level contents of /content/HAM10000:')
if base.exists():
    for p in sorted(base.iterdir()):
        print(' -', p.name)
else:
    print('Path does not exist yet.')


In [ ]:
# Cell 5: Create a small sampled dataset for smoke test
# Samples up to 50 images per class from the 7 HAM10000 classes.
import shutil
from pathlib import Path
import pandas as pd

DATA_DIR = Path('/content/HAM10000')
SAMPLE_DIR = Path('/content/HAM10000_sample')
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

metadata_csv = DATA_DIR / 'HAM10000_metadata.csv'
if not metadata_csv.exists():
    raise FileNotFoundError(f'Metadata CSV not found: {metadata_csv}')

class_names = ['akiec', 'bcc', 'bkl', 'df', 'nv', 'vasc', 'mel']

df = pd.read_csv(metadata_csv)
df = df[df['dx'].isin(class_names)].copy()

sampled = (
    df.groupby('dx', group_keys=False)
      .apply(lambda g: g.sample(n=min(len(g), 50), random_state=42))
      .reset_index(drop=True)
)

search_dirs = [
    DATA_DIR / 'HAM10000_images_part_1',
    DATA_DIR / 'HAM10000_images_part_2',
    DATA_DIR / 'HAM10000_images',
]
search_dirs = [d for d in search_dirs if d.exists()]
if not search_dirs:
    raise FileNotFoundError('No image folders found. Expected one of: HAM10000_images_part_1, HAM10000_images_part_2, HAM10000_images')

copied = 0
kept_rows = []
for _, row in sampled.iterrows():
    image_id = row['image_id']
    src = None
    for d in search_dirs:
        p = d / f'{image_id}.jpg'
        if p.exists():
            src = p
            break
    if src is None:
        continue
    dst = SAMPLE_DIR / f'{image_id}.jpg'
    shutil.copy2(src, dst)
    kept_rows.append(row)
    copied += 1

sample_df = pd.DataFrame(kept_rows)
out_csv = SAMPLE_DIR / 'metadata_sample.csv'
sample_df.to_csv(out_csv, index=False)

print('Sample dataset ready.')
print('Images copied:', copied)
print('metadata_sample.csv:', out_csv)
print(sample_df['dx'].value_counts().sort_index())


In [ ]:
# Cell 6: PyTorch smoke-test training (1 epoch)
# This is a quick verification run on the sampled subset from Cell 5.
from pathlib import Path
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'nv', 'vasc', 'mel']
CLASS_TO_INDEX = {cls: idx for idx, cls in enumerate(CLASS_NAMES)}

class SmallDataset(Dataset):
    def __init__(self, sample_dir, metadata_csv, transform=None):
        self.sample_dir = Path(sample_dir)
        self.transform = transform
        self.samples = []

        df = pd.read_csv(metadata_csv)
        df = df[df['dx'].isin(CLASS_NAMES)].copy()
        for _, row in df.iterrows():
            image_path = self.sample_dir / f"{row['image_id']}.jpg"
            if image_path.exists():
                self.samples.append((image_path, CLASS_TO_INDEX[row['dx']]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

sample_dir = Path('/content/HAM10000_sample')
metadata_csv = sample_dir / 'metadata_sample.csv'
if not metadata_csv.exists():
    raise FileNotFoundError('Run Cell 5 first to create metadata_sample.csv')

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

dataset = SmallDataset(sample_dir=sample_dir, metadata_csv=metadata_csv, transform=transform)
if len(dataset) == 0:
    raise RuntimeError('Sample dataset is empty. Check Cells 4-5.')

loader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

model = models.efficientnet_b0(weights='IMAGENET1K_V1')
in_features = model.classifier[1].in_features
# Keep classifier head structure aligned with repository train_pytorch.py
model.classifier = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(in_features, 512),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(256, len(CLASS_NAMES)),
)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3)

model.train()
running_loss = 0.0
correct = 0
total = 0

for images, labels in loader:
    images = images.to(device, non_blocking=True)
    labels = labels.to(device, non_blocking=True)

    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    running_loss += loss.item() * images.size(0)
    preds = outputs.argmax(dim=1)
    correct += (preds == labels).sum().item()
    total += labels.size(0)

epoch_loss = running_loss / total
epoch_acc = correct / total
print(f'Smoke-test epoch complete | loss: {epoch_loss:.4f} | acc: {epoch_acc:.4f}')

torch.save(model.state_dict(), '/content/efficientnet_smoke.pt')
print('Saved smoke-test model: /content/efficientnet_smoke.pt')


In [ ]:
# Cell 7: Full training instructions (using repository script)
# For full training, use the repository script after data is prepared in project root.
# In train_pytorch.py, use a portable project path:
#     PROJECT_DIR = Path(__file__).resolve().parent
# Example full training run (after cloning repo in Cell 8):
#     %cd /content/skin-disease-detection
#     !python train_pytorch.py
#
# Cell 6 is only a quick smoke test on a small sample.
print('See comments in this cell for full-training steps with train_pytorch.py.')


In [ ]:
# Cell 8: Optional clone command for this repository
!git clone https://github.com/Navanit018/skin-disease-detection.git /content/skin-disease-detection || true
!ls -la /content/skin-disease-detection | sed -n '1,200p'
